# COCO Small-Objects Local Prep + Google Drive Upload

This notebook runs **locally** and does the following:
1. Downloads COCO archives (idempotent: skips existing files).
2. Extracts archives (idempotent: skips existing folders/files).
3. Filters COCO annotations to keep only small objects (`area <= 32^2` by default).
4. Copies only required images.
5. Saves a compact COCO subset and optional ZIP.
6. Optionally uploads resulting ZIP/folder to Google Drive.


In [1]:
# Step 0 (optional): install dependencies.
# Uncomment and run once if needed.
%pip install -q tqdm pydrive2


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Step 1: imports and settings.
from __future__ import annotations

import json
import random
import shutil
import urllib.request
import zipfile
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from tqdm.auto import tqdm

# COCO official URLs
COCO_URLS = {
    'train2017.zip': 'http://images.cocodataset.org/zips/train2017.zip',
    'val2017.zip': 'http://images.cocodataset.org/zips/val2017.zip',
    'annotations_trainval2017.zip': 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
}

# Local workspace paths (adjust if needed)
PROJECT_ROOT = Path.cwd()
LOCAL_DATA_ROOT = PROJECT_ROOT / 'datasets_local' / 'coco_small_prep'
ARCHIVES_DIR = LOCAL_DATA_ROOT / 'archives'
RAW_DIR = LOCAL_DATA_ROOT / 'raw_coco2017'
SMALL_COCO_DIR = LOCAL_DATA_ROOT / 'coco_small_subset'
EXPORT_DIR = LOCAL_DATA_ROOT / 'export'

for p in (ARCHIVES_DIR, RAW_DIR, SMALL_COCO_DIR, EXPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Filtering settings
SMALL_MAX_AREA = 32.0**2
INCLUDE_CROWD = False
KEEP_IMAGES_WITHOUT_SMALL = False
MAX_TRAIN_IMAGES = None   # e.g. 50000 / 10000 / None
MAX_VAL_IMAGES = None     # e.g. 5000 / 2000 / None
RANDOM_SEED = 42

# Export settings
CREATE_EXPORT_ZIP = True
EXPORT_ZIP_NAME = 'coco_small_subset.zip'

# Upload settings
USE_DRIVE_SYNC_COPY = False
GOOGLE_DRIVE_SYNC_DIR = Path('C:/Users/your_user/Google Drive/My Drive/datasets')  # set your local Google Drive sync path

USE_PYDRIVE2_UPLOAD = False
PYDRIVE_PARENT_FOLDER_ID = None  # optional: Drive folder id

print('PROJECT_ROOT:', PROJECT_ROOT)
print('LOCAL_DATA_ROOT:', LOCAL_DATA_ROOT)


PROJECT_ROOT: d:\python_projects\vkr\small_objects_auto_aug\notebooks
LOCAL_DATA_ROOT: d:\python_projects\vkr\small_objects_auto_aug\notebooks\datasets_local\coco_small_prep


d:\python_projects\vkr\small_objects_auto_aug\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Step 2: download COCO archives (idempotent).
def download_file_if_missing(url: str, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        print(f'[SKIP] exists: {path.name} ({path.stat().st_size / (1024**3):.2f} GB)')
        return path
    print(f'[DL] {url} -> {path}')
    urllib.request.urlretrieve(url, str(path))
    print(f'[OK] downloaded: {path.name} ({path.stat().st_size / (1024**3):.2f} GB)')
    return path

archive_paths = {}
for name, url in COCO_URLS.items():
    archive_paths[name] = download_file_if_missing(url, ARCHIVES_DIR / name)

archive_paths


[DL] http://images.cocodataset.org/zips/train2017.zip -> d:\python_projects\vkr\small_objects_auto_aug\notebooks\datasets_local\coco_small_prep\archives\train2017.zip


ContentTooShortError: <urlopen error retrieval incomplete: got only 12460476392 out of 19336861798 bytes>

In [ ]:
# Step 3: extract archives (idempotent).
def extract_zip_if_needed(zip_path: Path, output_root: Path, readiness_check: list[Path]) -> None:
    if all(path.exists() for path in readiness_check):
        print(f'[SKIP] extraction already ready for {zip_path.name}')
        return
    print(f'[UNZIP] {zip_path.name}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(output_root)
    print(f'[OK] extracted {zip_path.name}')

extract_zip_if_needed(
    ARCHIVES_DIR / 'train2017.zip',
    RAW_DIR,
    [RAW_DIR / 'train2017'],
)
extract_zip_if_needed(
    ARCHIVES_DIR / 'val2017.zip',
    RAW_DIR,
    [RAW_DIR / 'val2017'],
)
extract_zip_if_needed(
    ARCHIVES_DIR / 'annotations_trainval2017.zip',
    RAW_DIR,
    [RAW_DIR / 'annotations' / 'instances_train2017.json', RAW_DIR / 'annotations' / 'instances_val2017.json'],
)

print('[OK] RAW_DIR ready:', RAW_DIR)


In [ ]:
# Step 4: helpers for filtering by small objects.
@dataclass
class SplitFilterReport:
    split: str
    images_total: int
    annotations_total: int
    images_kept: int
    annotations_kept: int
    categories_kept: int


def _load_json(path: Path) -> dict:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def _dump_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def _copy_image(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    shutil.copy2(src, dst)


def filter_coco_split_small(
    raw_root: Path,
    out_root: Path,
    split: str,
    small_max_area: float,
    include_crowd: bool,
    keep_images_without_small: bool,
    max_images: int | None,
    seed: int,
) -> SplitFilterReport:
    folder = f'{split}2017'
    ann_path = raw_root / 'annotations' / f'instances_{folder}.json'
    img_dir = raw_root / folder

    payload = _load_json(ann_path)
    images = payload.get('images', [])
    anns = payload.get('annotations', [])
    categories = payload.get('categories', [])

    images_by_id = {int(img['id']): img for img in images}
    anns_by_image = defaultdict(list)

    for ann in anns:
        if (not include_crowd) and int(ann.get('iscrowd', 0)) == 1:
            continue
        area = float(ann.get('area', 0.0))
        if area <= small_max_area:
            anns_by_image[int(ann['image_id'])].append(ann)

    kept_image_ids = []
    for image_id in images_by_id.keys():
        has_small = len(anns_by_image.get(image_id, [])) > 0
        if has_small or keep_images_without_small:
            kept_image_ids.append(image_id)

    if max_images is not None and len(kept_image_ids) > max_images:
        rng = random.Random(seed)
        kept_image_ids = sorted(rng.sample(kept_image_ids, max_images))

    kept_images = [images_by_id[i] for i in kept_image_ids]
    kept_annotations = []
    for image_id in kept_image_ids:
        kept_annotations.extend(anns_by_image.get(image_id, []))

    used_category_ids = {int(ann['category_id']) for ann in kept_annotations}
    kept_categories = [cat for cat in categories if int(cat['id']) in used_category_ids]

    out_img_dir = out_root / folder
    out_ann_dir = out_root / 'annotations'
    out_ann_dir.mkdir(parents=True, exist_ok=True)

    for img in tqdm(kept_images, desc=f'Copy {split} images'):
        src = img_dir / img['file_name']
        dst = out_img_dir / img['file_name']
        if not src.exists():
            continue
        _copy_image(src, dst)

    small_payload = {
        'info': payload.get('info', {}),
        'licenses': payload.get('licenses', []),
        'images': kept_images,
        'annotations': kept_annotations,
        'categories': kept_categories,
    }

    # Save both explicit "small" file and standard file name for compatibility.
    _dump_json(small_payload, out_ann_dir / f'instances_small_{folder}.json')
    _dump_json(small_payload, out_ann_dir / f'instances_{folder}.json')

    return SplitFilterReport(
        split=split,
        images_total=len(images),
        annotations_total=len(anns),
        images_kept=len(kept_images),
        annotations_kept=len(kept_annotations),
        categories_kept=len(kept_categories),
    )


In [ ]:
# Step 5: run filtering and save report.
# Idempotent behavior: existing output folder is cleaned once for a deterministic result.
if SMALL_COCO_DIR.exists():
    shutil.rmtree(SMALL_COCO_DIR)
SMALL_COCO_DIR.mkdir(parents=True, exist_ok=True)

train_report = filter_coco_split_small(
    raw_root=RAW_DIR,
    out_root=SMALL_COCO_DIR,
    split='train',
    small_max_area=SMALL_MAX_AREA,
    include_crowd=INCLUDE_CROWD,
    keep_images_without_small=KEEP_IMAGES_WITHOUT_SMALL,
    max_images=MAX_TRAIN_IMAGES,
    seed=RANDOM_SEED,
)

val_report = filter_coco_split_small(
    raw_root=RAW_DIR,
    out_root=SMALL_COCO_DIR,
    split='val',
    small_max_area=SMALL_MAX_AREA,
    include_crowd=INCLUDE_CROWD,
    keep_images_without_small=KEEP_IMAGES_WITHOUT_SMALL,
    max_images=MAX_VAL_IMAGES,
    seed=RANDOM_SEED + 1,
)

summary = {
    'small_max_area': SMALL_MAX_AREA,
    'include_crowd': INCLUDE_CROWD,
    'keep_images_without_small': KEEP_IMAGES_WITHOUT_SMALL,
    'max_train_images': MAX_TRAIN_IMAGES,
    'max_val_images': MAX_VAL_IMAGES,
    'reports': {
        'train': train_report.__dict__,
        'val': val_report.__dict__,
    },
    'small_coco_dir': str(SMALL_COCO_DIR),
}

report_path = SMALL_COCO_DIR / 'small_subset_report.json'
with report_path.open('w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('[OK] report:', report_path)
print(json.dumps(summary, ensure_ascii=False, indent=2)[:2000])


In [ ]:
# Step 6: optional ZIP export.
export_zip_path = EXPORT_DIR / EXPORT_ZIP_NAME
if CREATE_EXPORT_ZIP:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    if export_zip_path.exists():
        export_zip_path.unlink()
    print('[ZIP] creating:', export_zip_path)
    with zipfile.ZipFile(export_zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in SMALL_COCO_DIR.rglob('*'):
            if p.is_file():
                zf.write(p, arcname=p.relative_to(SMALL_COCO_DIR))
    print('[OK] zip size GB:', round(export_zip_path.stat().st_size / (1024**3), 3))
else:
    print('[SKIP] CREATE_EXPORT_ZIP=False')


In [ ]:
# Step 7A (optional): copy result into local Google Drive sync folder.
# Works if Google Drive for Desktop is installed and this path is synced.
if USE_DRIVE_SYNC_COPY:
    if not GOOGLE_DRIVE_SYNC_DIR.exists():
        raise FileNotFoundError(f'Google Drive sync dir not found: {GOOGLE_DRIVE_SYNC_DIR}')

    target_dir = GOOGLE_DRIVE_SYNC_DIR / SMALL_COCO_DIR.name
    if target_dir.exists():
        shutil.rmtree(target_dir)
    shutil.copytree(SMALL_COCO_DIR, target_dir)
    print('[OK] copied subset folder to Drive sync:', target_dir)

    if CREATE_EXPORT_ZIP and export_zip_path.exists():
        zip_target = GOOGLE_DRIVE_SYNC_DIR / export_zip_path.name
        shutil.copy2(export_zip_path, zip_target)
        print('[OK] copied zip to Drive sync:', zip_target)
else:
    print('[SKIP] USE_DRIVE_SYNC_COPY=False')


In [ ]:
# Step 7B (optional): upload ZIP to Google Drive via PyDrive2.
# Requires browser OAuth flow on first run.
if USE_PYDRIVE2_UPLOAD:
    from pydrive2.auth import GoogleAuth
    from pydrive2.drive import GoogleDrive

    if not CREATE_EXPORT_ZIP or not export_zip_path.exists():
        raise FileNotFoundError('ZIP export file is missing. Enable CREATE_EXPORT_ZIP and rerun Step 6.')

    gauth = GoogleAuth()
    gauth.LocalWebserverAuth()
    drive = GoogleDrive(gauth)

    metadata = {'title': export_zip_path.name}
    if PYDRIVE_PARENT_FOLDER_ID:
        metadata['parents'] = [{'id': PYDRIVE_PARENT_FOLDER_ID}]

    f = drive.CreateFile(metadata)
    f.SetContentFile(str(export_zip_path))
    f.Upload()
    print('[OK] uploaded file id:', f['id'])
else:
    print('[SKIP] USE_PYDRIVE2_UPLOAD=False')
